In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys

# ============================================================
# 0. 基本路径
# ============================================================
ROOT = Path(r"E:\GeoSIMCP_Reproduce\geoconformal-main")

print("=" * 80)
print("STEP 0 — China PM2.5 数据与源码审计")
print("=" * 80)
print("项目根目录存在:", ROOT.exists())
print("项目根目录:", ROOT)

# ============================================================
# 1. 搜索 exact PM2.5 数据
# ============================================================
target_name = "china_sites_2018_05_PM2.5_all.csv"
csv_matches = list(ROOT.rglob(target_name))

print("\n" + "=" * 80)
print("1. 搜索 exact PM2.5 CSV")
print("=" * 80)

if len(csv_matches) == 0:
    print("❌ 未找到:", target_name)
else:
    print(f"✅ 找到 {len(csv_matches)} 个:")
    for p in csv_matches:
        print("  ", p)

# ============================================================
# 2. 如果找到，检查数据
# ============================================================
if len(csv_matches) > 0:
    DATA_PATH = csv_matches[0]
    data = pd.read_csv(DATA_PATH)

    print("\n" + "=" * 80)
    print("2. 数据基本信息")
    print("=" * 80)

    print("读取文件:", DATA_PATH)
    print("Shape:", data.shape)

    print("\n全部字段:")
    print(list(data.columns))

    print("\n前 5 行:")
    display(data.head())

    # --------------------------------------------------------
    # 作者 GeoSIMCP 使用的核心变量
    # --------------------------------------------------------
    required = [
        "PM2.5",
        "dem", "sp", "tp", "blh", "r", "aod_sat", "ndvi",
        "lon", "lat"
    ]

    print("\n核心变量检查:")
    for col in required:
        print(f"{col:<10s} :", "✅" if col in data.columns else "❌ 缺失")

    # --------------------------------------------------------
    # date
    # --------------------------------------------------------
    if "date" in data.columns:
        print("\ndate 唯一值数量:", data["date"].nunique())
        print("date 唯一值前 20 个:")
        print(data["date"].unique()[:20])

    # --------------------------------------------------------
    # 缺失值 / inf
    # --------------------------------------------------------
    exist_required = [c for c in required if c in data.columns]

    print("\n核心变量缺失值:")
    print(data[exist_required].isna().sum())

    numeric = data[exist_required].select_dtypes(include=[np.number])

    print("\n核心数值变量 inf 数量:")
    print(
        pd.Series(
            np.isinf(numeric.values).sum(axis=0),
            index=numeric.columns
        )
    )

    print("\n核心变量描述统计:")
    display(data[exist_required].describe().T)

    # --------------------------------------------------------
    # 坐标 / 站点唯一性
    # --------------------------------------------------------
    if {"lon", "lat"}.issubset(data.columns):
        print("\n唯一坐标数:",
              data[["lon", "lat"]].drop_duplicates().shape[0])

    for station_col in ["station_id", "station", "Station", "site_id"]:
        if station_col in data.columns:
            print(
                f"{station_col} 唯一值数:",
                data[station_col].nunique()
            )

# ============================================================
# 3. 搜索作者 notebook
# ============================================================
print("\n" + "=" * 80)
print("3. 搜索 GeoSIMCP 作者 notebook")
print("=" * 80)

for nb_name in [
    "GeosimCP_real_world_all.ipynb",
    "GeosimCP_all.ipynb",
    "GeoSIMCP_real_world_all.ipynb"
]:
    matches = list(ROOT.rglob(nb_name))
    print(f"\n{nb_name}:")
    if matches:
        for p in matches:
            print("  ✅", p)
    else:
        print("  ❌ 未找到")

# ============================================================
# 4. 搜索作者原始 GeoCP / GeoSIMCP 源码
# ============================================================
print("\n" + "=" * 80)
print("4. 搜索作者原始源码")
print("=" * 80)

for filename in [
    "GeoConformalSpatialRegression.py",
    "GeoSIMCP.py",
    "GeoSIMCPzhao.py"
]:
    matches = list(ROOT.rglob(filename))

    print(f"\n{filename}:")
    if matches:
        for p in matches:
            print("  ✅", p)
    else:
        print("  ❌ 未找到")

print("\n" + "=" * 80)
print("STEP 0 完成")
print("=" * 80)

STEP 0 — China PM2.5 数据与源码审计
项目根目录存在: True
项目根目录: E:\GeoSIMCP_Reproduce\geoconformal-main

1. 搜索 exact PM2.5 CSV
❌ 未找到: china_sites_2018_05_PM2.5_all.csv

3. 搜索 GeoSIMCP 作者 notebook

GeosimCP_real_world_all.ipynb:
  ✅ E:\GeoSIMCP_Reproduce\geoconformal-main\Codes_Manuscript_GeoSIMCP\Codes_GeoSIMCP\GeosimCP_real_world_all.ipynb

GeosimCP_all.ipynb:
  ✅ E:\GeoSIMCP_Reproduce\geoconformal-main\Codes_Manuscript_GeoSIMCP\GeosimCP_all.ipynb

GeoSIMCP_real_world_all.ipynb:
  ✅ E:\GeoSIMCP_Reproduce\geoconformal-main\Codes_Manuscript_GeoSIMCP\Codes_GeoSIMCP\GeoSIMCP_real_world_all.ipynb

4. 搜索作者原始源码

GeoConformalSpatialRegression.py:
  ✅ E:\GeoSIMCP_Reproduce\geoconformal-main\build\lib\geoconformal\GeoConformalSpatialRegression.py
  ✅ E:\GeoSIMCP_Reproduce\geoconformal-main\Codes_Manuscript_GeoSIMCP\Codes_GeoSIMCP\GeoConformal\GeoConformalSpatialRegression.py
  ✅ E:\GeoSIMCP_Reproduce\geoconformal-main\geoconformal\GeoConformalSpatialRegression.py

GeoSIMCP.py:
  ✅ E:\GeoSIMCP_Reproduce\geoco

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM25\00_data"
    r"\china_sites_2018_05_PM2.5_all.csv"
)

print("文件存在:", DATA_PATH.exists())
print("文件路径:", DATA_PATH)

# Figshare 原始 CSV 是中文表头，优先尝试 GBK
try:
    data = pd.read_csv(DATA_PATH, encoding="gbk")
    print("读取编码: GBK")
except UnicodeDecodeError:
    data = pd.read_csv(DATA_PATH, encoding="utf-8")
    print("读取编码: UTF-8")

print("\nShape:", data.shape)

print("\n原始字段:")
print(data.columns.tolist())

print("\n前5行:")
display(data.head())

print("\n缺失值最多的字段:")
print(data.isna().sum().sort_values(ascending=False).head(15))

文件存在: False
文件路径: E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM25\00_data\china_sites_2018_05_PM2.5_all.csv


FileNotFoundError: [Errno 2] No such file or directory: 'E:\\GeoSIMCP_Reproduce\\geoconformal-main\\GCCP_PM25\\00_data\\china_sites_2018_05_PM2.5_all.csv'

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# STEP 0 — China PM2.5 数据审计
# ============================================================

DATA_PATH = Path("china_sites_2018_05_PM2.5_all.csv")

print("=" * 70)
print("STEP 0 — China PM2.5 数据审计")
print("=" * 70)

print("当前工作目录:", Path.cwd())
print("文件存在:", DATA_PATH.exists())
print("数据文件:", DATA_PATH.resolve())

# ------------------------------------------------------------
# 1. 尝试读取 CSV
# ------------------------------------------------------------
data = None

for enc in ["gbk", "gb18030", "utf-8", "utf-8-sig"]:
    try:
        data = pd.read_csv(DATA_PATH, encoding=enc)
        print("\n✅ 成功读取")
        print("编码:", enc)
        break
    except UnicodeDecodeError:
        continue

if data is None:
    raise RuntimeError("无法识别 CSV 编码")

# ------------------------------------------------------------
# 2. 基本信息
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("1. 基本信息")
print("=" * 70)

print("Shape:", data.shape)
print("总行数:", len(data))
print("总列数:", data.shape[1])

print("\n原始字段:")
for i, col in enumerate(data.columns, 1):
    print(f"{i:2d}. {col}")

print("\n前 5 行:")
display(data.head())

# ------------------------------------------------------------
# 3. 日期检查
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("2. 日期检查")
print("=" * 70)

if "date" in data.columns:
    print("date 唯一值数量:", data["date"].nunique())
    print("date 唯一值:", data["date"].unique())
else:
    print("❌ 没有找到 date 字段")

# ------------------------------------------------------------
# 4. 自动兼容中文/英文表头
# ------------------------------------------------------------
rename_map = {
    "监测点编码": "id",
    "监测点名称": "name",
    "城市": "city",
    "经度": "lon",
    "纬度": "lat",
}

data = data.rename(columns=rename_map)

print("\n" + "=" * 70)
print("3. 表头标准化之后")
print("=" * 70)

print(data.columns.tolist())

# ------------------------------------------------------------
# 5. GeoSIMCP 核心变量检查
# ------------------------------------------------------------
required = [
    "PM2.5",
    "dem",
    "sp",
    "tp",
    "blh",
    "r",
    "aod_sat",
    "ndvi",
    "lon",
    "lat"
]

print("\n" + "=" * 70)
print("4. GeoSIMCP 核心变量检查")
print("=" * 70)

all_ok = True

for col in required:
    if col in data.columns:
        print(f"✅ {col}")
    else:
        print(f"❌ {col} 缺失")
        all_ok = False

# ------------------------------------------------------------
# 6. 缺失值
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("5. 核心变量缺失值")
print("=" * 70)

existing = [c for c in required if c in data.columns]

print(data[existing].isna().sum())

# ------------------------------------------------------------
# 7. 无限值
# ------------------------------------------------------------
numeric = data[existing].select_dtypes(include=[np.number])

print("\n" + "=" * 70)
print("6. 核心变量 Inf 检查")
print("=" * 70)

inf_count = pd.Series(
    np.isinf(numeric.to_numpy()).sum(axis=0),
    index=numeric.columns
)

print(inf_count)

# ------------------------------------------------------------
# 8. 唯一站点 / 坐标
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("7. 站点与坐标检查")
print("=" * 70)

if "id" in data.columns:
    print("站点 ID 数:", data["id"].nunique())
    print("重复站点 ID:", data["id"].duplicated().sum())

if {"lon", "lat"}.issubset(data.columns):
    print(
        "唯一经纬度数:",
        data[["lon", "lat"]].drop_duplicates().shape[0]
    )

# ------------------------------------------------------------
# 9. 数值统计
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("8. 核心变量统计")
print("=" * 70)

display(data[existing].describe().T)

# ------------------------------------------------------------
# 10. 最终判断
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 0 初步结论")
print("=" * 70)

if data.shape == (1415, 24):
    print("✅ Shape 与 GeoSIMCP notebook 一致: 1415 × 24")
else:
    print("⚠️ Shape 与预期不同:", data.shape)

if all_ok:
    print("✅ GeoSIMCP 所需核心变量全部存在")
else:
    print("❌ 存在核心变量缺失")

print("\nSTEP 0 运行完成")

STEP 0 — China PM2.5 数据审计
当前工作目录: E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\00_data_audit
文件存在: True
数据文件: E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\00_data_audit\china_sites_2018_05_PM2.5_all.csv

✅ 成功读取
编码: gbk

1. 基本信息
Shape: (1415, 24)
总行数: 1415
总列数: 24

原始字段:
 1. 监测点编码
 2. 监测点名称
 3. 城市
 4. 经度
 5. 纬度
 6. date
 7. PM2.5
 8. row_index
 9. col_index
10. proj_x
11. proj_y
12. dem
13. w10
14. d10
15. t2m
16. sp
17. tp
18. blh
19. e
20. r
21. u10
22. v10
23. aod_sat
24. ndvi

前 5 行:


,监测点编码,监测点名称,城市,经度,纬度,date,PM2.5,row_index,col_index,proj_x,...,t2m,sp,tp,blh,e,r,u10,v10,aod_sat,ndvi
0,1001A,万寿西宫,北京,116.366,39.8673,20180531,60.548387,2201.0,6867.0,1.650848e+06,...,293.967224,100044.187500,0.000634,164.847870,-0.000010,44.939587,0.764578,1.060548,0.660040,3932
1,1002A,定陵,北京,116.170,40.2865,20180531,57.224138,2134.0,6835.0,1.625004e+06,...,291.590820,96484.054688,0.000450,101.149689,-0.000010,51.034016,0.007471,-0.032906,0.568065,7575
2,1003A,东四,北京,116.434,39.9522,20180531,67.258065,2188.0,6877.0,1.653777e+06,...,294.037201,100061.195312,0.000598,163.240631,-0.000009,44.052452,0.581442,1.025983,0.683423,3769
3,1004A,天坛,北京,116.434,39.8745,20180531,57.080645,2200.0,6877.0,1.655828e+06,...,294.102112,100167.195312,0.000655,175.418289,-0.000011,44.226921,0.670114,1.145302,0.653063,4337
4,1005A,农展馆,北京,116.473,39.9716,20180531,57.935484,2185.0,6884.0,1.656225e+06,...,294.140289,100108.039062,0.000601,169.096985,-0.000009,43.352154,0.494377,1.053341,0.653476,3757



2. 日期检查
date 唯一值数量: 1
date 唯一值: [20180531]

3. 表头标准化之后
['id', 'name', 'city', 'lon', 'lat', 'date', 'PM2.5', 'row_index', 'col_index', 'proj_x', 'proj_y', 'dem', 'w10', 'd10', 't2m', 'sp', 'tp', 'blh', 'e', 'r', 'u10', 'v10', 'aod_sat', 'ndvi']

4. GeoSIMCP 核心变量检查
✅ PM2.5
✅ dem
✅ sp
✅ tp
✅ blh
✅ r
✅ aod_sat
✅ ndvi
✅ lon
✅ lat

5. 核心变量缺失值
PM2.5      0
dem        0
sp         0
tp         0
blh        0
r          0
aod_sat    0
ndvi       0
lon        0
lat        0
dtype: int64

6. 核心变量 Inf 检查
PM2.5      0
dem        0
sp         0
tp         0
blh        0
r          0
aod_sat    0
ndvi       0
lon        0
lat        0
dtype: int64

7. 站点与坐标检查
站点 ID 数: 1415
重复站点 ID: 0
唯一经纬度数: 1415

8. 核心变量统计


,count,mean,std,min,25%,50%,75%,max
PM2.5,1415.0,32.983795,12.763498,3.112903,25.426344,32.189655,39.975806,191.620690
dem,1415.0,398.286219,663.337535,-5.000000,29.000000,97.000000,439.000000,4520.000000
sp,1415.0,95963.543104,7434.050694,57284.242188,94716.242188,99361.093750,100467.351562,101184.343750
tp,1415.0,0.002210,0.001355,0.000062,0.001026,0.002059,0.003244,0.007347
blh,1415.0,246.886057,131.718369,59.342499,181.514748,232.939743,282.254578,2278.236084
r,1415.0,65.648181,15.949851,15.003161,54.262581,71.049835,77.903458,93.906235
aod_sat,1415.0,0.561301,0.169551,0.067142,0.447181,0.571641,0.684930,0.895976
ndvi,1415.0,3326.983039,1171.374248,26.000000,2567.000000,3098.000000,3811.000000,8672.000000
lon,1415.0,113.323814,8.444840,75.943500,109.062800,114.286000,118.829000,131.165100
lat,1415.0,33.186432,6.357294,19.950700,28.593300,32.188889,37.822250,50.426900



STEP 0 初步结论
✅ Shape 与 GeoSIMCP notebook 一致: 1415 × 24
✅ GeoSIMCP 所需核心变量全部存在

STEP 0 运行完成


In [4]:
from pathlib import Path
import json

ROOT = Path(r"E:\GeoSIMCP_Reproduce\geoconformal-main")

AUTHOR_NB = (
    ROOT
    / "Codes_Manuscript_GeoSIMCP"
    / "Codes_GeoSIMCP"
    / "GeosimCP_real_world_all.ipynb"
)

print("=" * 80)
print("AUTHOR SOURCE AUDIT — China PM2.5")
print("=" * 80)
print("Notebook 存在:", AUTHOR_NB.exists())
print("Notebook:", AUTHOR_NB)

with open(AUTHOR_NB, "r", encoding="utf-8") as f:
    nb = json.load(f)

cells = nb["cells"]

# ------------------------------------------------------------
# 找 China PM2.5 起始 cell
# ------------------------------------------------------------
pm25_idx = None

for i, cell in enumerate(cells):
    src = "".join(cell.get("source", []))
    if "China PM2.5" in src or "china_sites_2018_05_PM2.5_all.csv" in src:
        pm25_idx = i
        break

print("\nPM2.5 起始 cell:", pm25_idx)

if pm25_idx is None:
    raise RuntimeError("没有在作者 notebook 中定位到 China PM2.5")

# ------------------------------------------------------------
# 只检查 PM2.5 后面的相关代码
# ------------------------------------------------------------
keywords = [
    "china_sites_2018_05_PM2.5_all.csv",
    "features",
    "X_aspatial",
    "X_spatial",
    "loc =",
    "train_test_split",
    "random_state",
    "train_size",
    "test_size",
    "RandomForestRegressor",
    "GridSearchCV",
    "param_grid",
    "n_estimators",
    "max_depth",
    "best_params_",
    "best_estimator_",
    "model_spatial",
    "mean_squared_error",
    "mean_absolute_error",
    "r2_score",
    "RMSE",
    "MAE",
]

print("\n" + "=" * 80)
print("PM2.5 相关作者源码")
print("=" * 80)

# 往后检查足够多 cells
end_idx = min(pm25_idx + 100, len(cells))

for i in range(pm25_idx, end_idx):
    cell = cells[i]

    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))

    if any(k in src for k in keywords):
        print("\n" + "#" * 80)
        print(f"CELL {i}")
        print("#" * 80)
        print(src)

print("\n" + "=" * 80)
print("AUTHOR SOURCE AUDIT 完成")
print("=" * 80)

AUTHOR SOURCE AUDIT — China PM2.5
Notebook 存在: True
Notebook: E:\GeoSIMCP_Reproduce\geoconformal-main\Codes_Manuscript_GeoSIMCP\Codes_GeoSIMCP\GeosimCP_real_world_all.ipynb

PM2.5 起始 cell: 11

PM2.5 相关作者源码

################################################################################
CELL 11
################################################################################
# # ## China PM2.5
data = pd.read_csv('/content/drive/MyDrive/Research/0_GeoCP_with_geosimilarity//data//china_sites_2018_05_PM2.5_all.csv')

features = data[['dem', 'sp', 'tp', 'blh', 'r', 'aod_sat', 'ndvi']]
loc = data[['lon', 'lat']]

y = data['PM2.5']
X_aspatial = data[['dem', 'sp', 'tp', 'blh', 'r', 'aod_sat', 'ndvi']]
X_spatial = data[['dem', 'sp', 'tp', 'blh', 'r', 'aod_sat', 'ndvi','lon', 'lat']]


################################################################################
CELL 14
################################################################################
import numpy as np
from sklearn.metrics imp

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ============================================================
# STEP 1 — Author-aligned Random Forest reproduction
# ============================================================

ROOT = Path(r"E:\GeoSIMCP_Reproduce\geoconformal-main")

DATA_PATH = (
    ROOT
    / "GCCP_PM2.5"
    / "00_data_audit"
    / "china_sites_2018_05_PM2.5_all.csv"
)

OUTPUT_DIR = ROOT / "GCCP_PM2.5" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("STEP 1 — Author Random Forest reproduction")
print("=" * 80)

# ------------------------------------------------------------
# 1. 读取数据
# ------------------------------------------------------------
data = pd.read_csv(DATA_PATH, encoding="gbk")

data = data.rename(
    columns={
        "监测点编码": "id",
        "监测点名称": "name",
        "城市": "city",
        "经度": "lon",
        "纬度": "lat",
    }
)

FEATURES = [
    "dem",
    "sp",
    "tp",
    "blh",
    "r",
    "aod_sat",
    "ndvi",
]

X_aspatial = data[FEATURES].copy()

X_spatial = data[
    FEATURES + ["lon", "lat"]
].copy()

loc = data[["lon", "lat"]].copy()
y = data["PM2.5"].copy()

print("Data shape:", data.shape)
print("Predictor shape:", X_spatial.shape)

# ------------------------------------------------------------
# 2. 完全按照作者源码划分
# ------------------------------------------------------------
(
    X_aspatial_train,
    X_apsatial_temp,       # 保留作者源码中的拼写
    X_spatial_train,
    X_spatial_temp,
    y_train,
    y_temp,
    loc_train,
    loc_temp,
) = train_test_split(
    X_aspatial,
    X_spatial,
    y,
    loc,
    train_size=0.8,
    random_state=42,
)

(
    X_aspatial_val,
    X_aspatial_test,
    X_spatial_val,
    X_spatial_test,
    y_val,
    y_test,
    loc_val,
    loc_test,
) = train_test_split(
    X_apsatial_temp,
    X_spatial_temp,
    y_temp,
    loc_temp,
    train_size=0.5,
    random_state=42,
)

print("\n" + "=" * 80)
print("AUTHOR SPLIT")
print("=" * 80)

print("Train      :", len(y_train))
print("Validation :", len(y_val))
print("Test       :", len(y_test))
print("Total      :", len(y_train) + len(y_val) + len(y_test))

# ------------------------------------------------------------
# 3. 作者 Random Forest
# ------------------------------------------------------------
rf = RandomForestRegressor(
    random_state=42
)

param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [None, 10],
}

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    return_train_score=True,
)

print("\n开始 Random Forest GridSearchCV ...")
grid.fit(X_spatial_train, y_train)

model_spatial = grid

print("\n" + "=" * 80)
print("GRID SEARCH RESULT")
print("=" * 80)

print("Best parameters:", grid.best_params_)
print("Best CV negative MSE:", grid.best_score_)
print(
    "Best CV RMSE:",
    np.sqrt(-grid.best_score_)
)

# ------------------------------------------------------------
# 4. Validation / Test 点预测
# ------------------------------------------------------------
y_val_pred = model_spatial.predict(X_spatial_val)
y_test_pred = model_spatial.predict(X_spatial_test)

def regression_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2

rmse_val, mae_val, r2_val = regression_metrics(
    y_val,
    y_val_pred
)

rmse_test, mae_test, r2_test = regression_metrics(
    y_test,
    y_test_pred
)

print("\n" + "=" * 80)
print("POINT PREDICTION PERFORMANCE")
print("=" * 80)

print(
    f"[Validation] "
    f"RMSE={rmse_val:.6f}, "
    f"MAE={mae_val:.6f}, "
    f"R²={r2_val:.6f}"
)

print(
    f"[Test]       "
    f"RMSE={rmse_test:.6f}, "
    f"MAE={mae_test:.6f}, "
    f"R²={r2_test:.6f}"
)

# ------------------------------------------------------------
# 5. 和正式论文结果作参照
#    这里只比较，不用这些数字调模型
# ------------------------------------------------------------
REF_RMSE = 4.711
REF_MAE  = 3.648
REF_R2   = 0.802

print("\n" + "=" * 80)
print("REFERENCE CHECK")
print("=" * 80)

print("正式论文 Random Forest Test:")
print(
    f"RMSE={REF_RMSE}, "
    f"MAE={REF_MAE}, "
    f"R²={REF_R2}"
)

print("\n当前复现 - 论文:")
print(f"ΔRMSE = {rmse_test - REF_RMSE:+.6f}")
print(f"ΔMAE  = {mae_test - REF_MAE:+.6f}")
print(f"ΔR²   = {r2_test - REF_R2:+.6f}")

# ------------------------------------------------------------
# 6. 保存预测结果
# ------------------------------------------------------------
pred_df = data.loc[y_test.index, [
    "id", "name", "city", "lon", "lat", "PM2.5"
]].copy()

pred_df["y_pred"] = y_test_pred
pred_df["residual"] = (
    pred_df["PM2.5"] - pred_df["y_pred"]
)
pred_df["abs_residual"] = np.abs(
    pred_df["residual"]
)

PRED_PATH = OUTPUT_DIR / "pm25_author_rf_test_predictions.csv"

pred_df.to_csv(
    PRED_PATH,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 7. 保存摘要
# ------------------------------------------------------------
summary_df = pd.DataFrame([{
    "split": "author_80_10_10_random_state42",
    "model": "RandomForestRegressor",
    "best_n_estimators": grid.best_params_["n_estimators"],
    "best_max_depth": str(grid.best_params_["max_depth"]),
    "cv": 3,
    "validation_rmse": rmse_val,
    "validation_mae": mae_val,
    "validation_r2": r2_val,
    "test_rmse": rmse_test,
    "test_mae": mae_test,
    "test_r2": r2_test,
}])

SUMMARY_PATH = OUTPUT_DIR / "pm25_author_rf_summary.csv"

summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 80)
print("OUTPUT")
print("=" * 80)

print("预测结果:", PRED_PATH)
print("摘要结果:", SUMMARY_PATH)

print("\n✅ STEP 1 完成")

STEP 1 — Author Random Forest reproduction
Data shape: (1415, 24)
Predictor shape: (1415, 9)

AUTHOR SPLIT
Train      : 1132
Validation : 141
Test       : 142
Total      : 1415

开始 Random Forest GridSearchCV ...

GRID SEARCH RESULT
Best parameters: {'max_depth': None, 'n_estimators': 300}
Best CV negative MSE: -52.442028390639564
Best CV RMSE: 7.241686847043274

POINT PREDICTION PERFORMANCE
[Validation] RMSE=7.621597, MAE=4.683721, R²=0.787264
[Test]       RMSE=4.711488, MAE=3.647601, R²=0.802092

REFERENCE CHECK
正式论文 Random Forest Test:
RMSE=4.711, MAE=3.648, R²=0.802

当前复现 - 论文:
ΔRMSE = +0.000488
ΔMAE  = -0.000399
ΔR²   = +0.000092

OUTPUT
预测结果: E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_author_rf_test_predictions.csv
摘要结果: E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_author_rf_summary.csv

✅ STEP 1 完成


In [6]:
# ============================================================
# STEP 2 — Author UQ baseline reproduction
#
# GeoCP
# GeoSIMCP-EUC
# GeoSIMCP-MND (Zhao)
#
# 注意：
# 本 STEP 为“作者结果复现”
# 参数选择会按照作者源码使用 Test y
# 后续我们自己的正式 GCCP 实验绝不这样做
# ============================================================

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import time

# ------------------------------------------------------------
# 0. 作者源码路径
# ------------------------------------------------------------

ROOT = Path(r"E:\GeoSIMCP_Reproduce\geoconformal-main")

AUTHOR_CODE = (
    ROOT
    / "Codes_Manuscript_GeoSIMCP"
    / "Codes_GeoSIMCP"
    / "GeoConformal"
)

OUTPUT_DIR = ROOT / "GCCP_PM2.5" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(AUTHOR_CODE) not in sys.path:
    sys.path.insert(0, str(AUTHOR_CODE))

print("=" * 80)
print("STEP 2 — Author UQ baseline reproduction")
print("=" * 80)

print("Author code exists:", AUTHOR_CODE.exists())
print("Author code path:", AUTHOR_CODE)

# ------------------------------------------------------------
# 1. 导入作者原始实现
# ------------------------------------------------------------

from GeoConformalSpatialRegression import GeoConformalSpatialRegression
from GeoSIMCP import GeoSIMConformalSpatialRegression
from GeoSIMCPzhao import GeoSIMCPzhao

print("\n✅ 作者 GeoCP / GeoSIMCP / GeoSIMCP-MND 源码导入成功")

# ------------------------------------------------------------
# 2. 确认 STEP 1 的变量仍然存在
# ------------------------------------------------------------

required_names = [
    "model_spatial",
    "X_spatial_val",
    "X_spatial_test",
    "X_aspatial_val",
    "X_aspatial_test",
    "y_val",
    "y_test",
    "loc_val",
    "loc_test",
]

missing = [
    name for name in required_names
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "缺少 STEP 1 变量，请先重新运行 STEP 1："
        + ", ".join(missing)
    )

print("✅ STEP 1 数据和 Random Forest 已存在")

# ------------------------------------------------------------
# 3. 先检查 MND 的 Calibration feature range
# ------------------------------------------------------------

feature_ranges = (
    X_aspatial_val.max(axis=0)
    - X_aspatial_val.min(axis=0)
)

print("\n" + "=" * 80)
print("CALIBRATION FEATURE RANGE CHECK")
print("=" * 80)

print(feature_ranges)

zero_range = feature_ranges[
    np.isclose(feature_ranges.to_numpy(dtype=float), 0.0)
]

if len(zero_range) == 0:
    print("\n✅ Calibration 中不存在 zero-range feature")
else:
    print("\n⚠️ Zero-range features:")
    print(zero_range)

# ------------------------------------------------------------
# 4. Interval Score
# ------------------------------------------------------------

ALPHA = 0.1
COVERAGE_THRESHOLD = 0.9

def interval_score(y_true, lower, upper, alpha=0.1):
    """
    Interval Score（区间得分）:
    区间越窄越好，但漏掉真实值会受到额外惩罚。
    越小越好。
    """
    y_true = np.asarray(y_true, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)

    width = upper - lower

    below = (
        (lower - y_true)
        * (y_true < lower)
    )

    above = (
        (y_true - upper)
        * (y_true > upper)
    )

    return (
        width
        + (2.0 / alpha) * below
        + (2.0 / alpha) * above
    )


def summarize_result(
    result,
    y_true,
    method,
    bandwidth=None,
    lambda_weight=None,
):
    """
    同时明确区分：
    1. Mean Half-width = mean(q)
    2. Mean Width      = mean(upper-lower)

    防止作者代码中的 “Mean Interval” 命名歧义。
    """

    lower = np.asarray(
        result.lower_bound,
        dtype=float
    )

    upper = np.asarray(
        result.upper_bound,
        dtype=float
    )

    q = np.asarray(
        result.geo_uncertainty,
        dtype=float
    )

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    coverage = np.mean(
        (y_true >= lower)
        & (y_true <= upper)
    )

    mean_half_width = np.nanmean(q)

    mean_width = np.nanmean(
        upper - lower
    )

    mean_score = np.nanmean(
        interval_score(
            y_true,
            lower,
            upper,
            alpha=ALPHA
        )
    )

    return {
        "method": method,
        "bandwidth": bandwidth,
        "lambda": lambda_weight,
        "coverage": coverage,
        "mean_half_width": mean_half_width,
        "mean_width": mean_width,
        "interval_score": mean_score,
    }

# ============================================================
# PART A — GeoCP
# ============================================================

print("\n" + "=" * 80)
print("A. GeoCP")
print("=" * 80)

bandwidth_list = np.linspace(
    0.1,
    5.0,
    20
)

geocp_rows = []
geocp_results = []

start = time.time()

for bw in bandwidth_list:

    uq = GeoConformalSpatialRegression(
        predict_f=model_spatial.predict,
        nonconformity_score_f=None,
        miscoverage_level=ALPHA,
        bandwidth=bw,
        coord_calib=loc_val.values,
        coord_test=loc_test.values,
        X_calib=X_spatial_val,
        y_calib=y_val,
        X_test=X_spatial_test,
        y_test=y_test,
    )

    result = uq.analyze()

    row = summarize_result(
        result=result,
        y_true=y_test,
        method="GeoCP",
        bandwidth=bw,
    )

    geocp_rows.append(row)
    geocp_results.append(result)

geocp_df = pd.DataFrame(geocp_rows)

mask = (
    geocp_df["coverage"]
    >= COVERAGE_THRESHOLD
)

valid = geocp_df.loc[mask]

if len(valid) == 0:
    raise RuntimeError(
        "GeoCP 没有任何 Coverage >= 0.9 的参数"
    )

best_index = (
    valid["interval_score"]
    .idxmin()
)

best_geocp_row = geocp_df.loc[
    best_index
].copy()

best_geocp_result = geocp_results[
    best_index
]

print("\nGeoCP best:")
print(best_geocp_row)

print(
    f"\nGeoCP time: "
    f"{time.time() - start:.2f} sec"
)

# ============================================================
# PART B — GeoSIMCP-EUC
# ============================================================

print("\n" + "=" * 80)
print("B. GeoSIMCP-EUC")
print("=" * 80)

bandwidth_list = np.linspace(
    0.1,
    5.0,
    20
)

lambda_list = np.linspace(
    0.0,
    1.0,
    20
)

euc_rows = []
euc_results = []

start = time.time()

for bw in bandwidth_list:

    for lam in lambda_list:

        uq = GeoSIMConformalSpatialRegression(
            predict_f=model_spatial.predict,
            miscoverage_level=ALPHA,
            bandwidth=bw,
            coord_calib=loc_val.values,
            coord_test=loc_test.values,

            # 点预测模型使用 7 features + lon/lat
            X_calib=X_spatial_val,
            y_calib=y_val,

            X_test=X_spatial_test,
            y_test=y_test,

            # similarity 只使用 7 个非空间变量
            X_calib_weight=X_aspatial_val,
            X_test_weight=X_aspatial_test,

            lambda_weight=lam,
        )

        result = uq.analyze()

        row = summarize_result(
            result=result,
            y_true=y_test,
            method="GeoSIMCP-EUC",
            bandwidth=bw,
            lambda_weight=lam,
        )

        euc_rows.append(row)
        euc_results.append(result)

euc_df = pd.DataFrame(euc_rows)

mask = (
    euc_df["coverage"]
    >= COVERAGE_THRESHOLD
)

valid = euc_df.loc[mask]

if len(valid) == 0:
    raise RuntimeError(
        "GeoSIMCP-EUC 没有 Coverage >= 0.9 的参数"
    )

best_index = (
    valid["interval_score"]
    .idxmin()
)

best_euc_row = euc_df.loc[
    best_index
].copy()

best_euc_result = euc_results[
    best_index
]

print("\nGeoSIMCP-EUC best:")
print(best_euc_row)

print(
    f"\nGeoSIMCP-EUC time: "
    f"{time.time() - start:.2f} sec"
)

# ============================================================
# PART C — GeoSIMCP-MND / Zhao
# ============================================================

print("\n" + "=" * 80)
print("C. GeoSIMCP-MND")
print("=" * 80)

mnd_rows = []
mnd_results = []

start = time.time()

for bw in bandwidth_list:

    for lam in lambda_list:

        uq = GeoSIMCPzhao(
            predict_f=model_spatial.predict,
            miscoverage_level=ALPHA,
            bandwidth=bw,
            coord_calib=loc_val.values,
            coord_test=loc_test.values,

            X_calib=X_spatial_val,
            y_calib=y_val,

            X_test=X_spatial_test,
            y_test=y_test,

            X_calib_weight=X_aspatial_val,
            X_test_weight=X_aspatial_test,

            lambda_weight=lam,
        )

        result = uq.analyze()

        row = summarize_result(
            result=result,
            y_true=y_test,
            method="GeoSIMCP-MND",
            bandwidth=bw,
            lambda_weight=lam,
        )

        mnd_rows.append(row)
        mnd_results.append(result)

mnd_df = pd.DataFrame(mnd_rows)

mask = (
    mnd_df["coverage"]
    >= COVERAGE_THRESHOLD
)

valid = mnd_df.loc[mask]

if len(valid) == 0:
    raise RuntimeError(
        "GeoSIMCP-MND 没有 Coverage >= 0.9 的参数"
    )

best_index = (
    valid["interval_score"]
    .idxmin()
)

best_mnd_row = mnd_df.loc[
    best_index
].copy()

best_mnd_result = mnd_results[
    best_index
]

print("\nGeoSIMCP-MND best:")
print(best_mnd_row)

print(
    f"\nGeoSIMCP-MND time: "
    f"{time.time() - start:.2f} sec"
)

# ============================================================
# 5. 汇总
# ============================================================

summary = pd.DataFrame([
    best_geocp_row,
    best_euc_row,
    best_mnd_row,
])

print("\n" + "=" * 80)
print("AUTHOR UQ REPRODUCTION SUMMARY")
print("=" * 80)

display(summary)

# ------------------------------------------------------------
# 6. 参考作者正式结果
#
# 注意：
# 作者报告中的 “Mean Interval”
# 实际更接近 mean(q)，也就是 half-width。
# 所以这里和 mean_half_width 对照。
# ------------------------------------------------------------

reference = pd.DataFrame([
    {
        "method": "GeoCP",
        "bandwidth_ref": 2.937,
        "lambda_ref": np.nan,
        "coverage_ref": 0.937,
        "interval_score_ref": 24.99,
        "mean_half_width_ref": 10.56,
    },

    {
        "method": "GeoSIMCP-EUC",
        "bandwidth_ref": 2.679,
        "lambda_ref": 0.000,
        "coverage_ref": 0.965,
        "interval_score_ref": 21.45,
        "mean_half_width_ref": 9.64,
    },

    {
        "method": "GeoSIMCP-MND",
        "bandwidth_ref": 0.616,
        "lambda_ref": 0.000,
        "coverage_ref": 0.958,
        "interval_score_ref": 21.75,
        "mean_half_width_ref": 10.11,
    },
])

comparison = summary.merge(
    reference,
    on="method",
    how="left",
)

comparison["delta_coverage"] = (
    comparison["coverage"]
    - comparison["coverage_ref"]
)

comparison["delta_interval_score"] = (
    comparison["interval_score"]
    - comparison["interval_score_ref"]
)

comparison["delta_mean_half_width"] = (
    comparison["mean_half_width"]
    - comparison["mean_half_width_ref"]
)

print("\n" + "=" * 80)
print("REFERENCE COMPARISON")
print("=" * 80)

display(comparison)

# ============================================================
# 7. 保存所有网格 + 最优结果
# ============================================================

geocp_df.to_csv(
    OUTPUT_DIR
    / "pm25_author_geocp_grid.csv",
    index=False,
    encoding="utf-8-sig",
)

euc_df.to_csv(
    OUTPUT_DIR
    / "pm25_author_geosimcp_euc_grid.csv",
    index=False,
    encoding="utf-8-sig",
)

mnd_df.to_csv(
    OUTPUT_DIR
    / "pm25_author_geosimcp_mnd_grid.csv",
    index=False,
    encoding="utf-8-sig",
)

summary.to_csv(
    OUTPUT_DIR
    / "pm25_author_uq_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

comparison.to_csv(
    OUTPUT_DIR
    / "pm25_author_uq_reference_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n保存完成:")
print(
    OUTPUT_DIR
    / "pm25_author_uq_summary.csv"
)

print(
    OUTPUT_DIR
    / "pm25_author_uq_reference_comparison.csv"
)

print("\n✅ STEP 2 完成")

STEP 2 — Author UQ baseline reproduction
Author code exists: True
Author code path: E:\GeoSIMCP_Reproduce\geoconformal-main\Codes_Manuscript_GeoSIMCP\Codes_GeoSIMCP\GeoConformal

✅ 作者 GeoCP / GeoSIMCP / GeoSIMCP-MND 源码导入成功
✅ STEP 1 数据和 Random Forest 已存在

CALIBRATION FEATURE RANGE CHECK
dem         3847.000000
sp         38818.367188
tp             0.007086
blh          904.567276
r             70.259323
aod_sat        0.756998
ndvi        7443.000000
dtype: float64

✅ Calibration 中不存在 zero-range feature

A. GeoCP


E:\GeoSIMCP_Reproduce\geoconformal-main\Codes_Manuscript_GeoSIMCP\Codes_GeoSIMCP\GeoConformal\GeoConformalSpatialRegression.py:131: RuntimeWarning: invalid value encountered in divide
  normalized_cumsum_weights = cumsum_weights / cumsum_weights[-1]



GeoCP best:
method                 GeoCP
bandwidth           2.936842
lambda                  None
coverage             0.93662
mean_half_width    10.558143
mean_width         21.116286
interval_score      24.99337
Name: 11, dtype: object

GeoCP time: 1.25 sec

B. GeoSIMCP-EUC

GeoSIMCP-EUC best:
method             GeoSIMCP-EUC
bandwidth              2.678947
lambda                      0.0
coverage               0.964789
mean_half_width        9.638016
mean_width            19.276033
interval_score        21.448146
Name: 200, dtype: object

GeoSIMCP-EUC time: 24.02 sec

C. GeoSIMCP-MND

GeoSIMCP-MND best:
method             GeoSIMCP-MND
bandwidth              0.615789
lambda                      0.0
coverage               0.957746
mean_half_width       10.111825
mean_width             20.22365
interval_score        21.749173
Name: 40, dtype: object

GeoSIMCP-MND time: 24.21 sec

AUTHOR UQ REPRODUCTION SUMMARY


,method,bandwidth,lambda,coverage,mean_half_width,mean_width,interval_score
11,GeoCP,2.936842,NaN,0.936620,10.558143,21.116286,24.993370
200,GeoSIMCP-EUC,2.678947,0.0,0.964789,9.638016,19.276033,21.448146
40,GeoSIMCP-MND,0.615789,0.0,0.957746,10.111825,20.223650,21.749173



REFERENCE COMPARISON


,method,bandwidth,lambda,coverage,mean_half_width,mean_width,interval_score,bandwidth_ref,lambda_ref,coverage_ref,interval_score_ref,mean_half_width_ref,delta_coverage,delta_interval_score,delta_mean_half_width
0,GeoCP,2.936842,NaN,0.936620,10.558143,21.116286,24.993370,2.937,NaN,0.937,24.99,10.56,-0.000380,0.003370,-0.001857
1,GeoSIMCP-EUC,2.678947,0.0,0.964789,9.638016,19.276033,21.448146,2.679,0.0,0.965,21.45,9.64,-0.000211,-0.001854,-0.001984
2,GeoSIMCP-MND,0.615789,0.0,0.957746,10.111825,20.223650,21.749173,0.616,0.0,0.958,21.75,10.11,-0.000254,-0.000827,0.001825



保存完成:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_author_uq_summary.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_author_uq_reference_comparison.csv

✅ STEP 2 完成
